# Test d'enrichissement géographique — US Census API

**Objectif** : valider la récupération des 3 métriques Census par ZCTA (ZIP Code Tabulation Area) :
- Revenu médian des ménages (`B19013_001E`)
- Âge médian (`B01002_001E`)
- Densité de population (population / superficie terrestre en km²)

**Source** : ACS 5-Year 2022 + TIGER/Web REST API pour les superficies  
**Clé API** : gratuite sur https://api.census.gov/data/key_signup.html  
**Dépendances** : uniquement `urllib` (stdlib) + `pandas`

## 0. Configuration

In [ ]:
import urllib.request
import urllib.parse
import json
import pandas as pd
import time

# ─── Clé API Census (gratuite : https://api.census.gov/data/key_signup.html) ───
CENSUS_API_KEY = "8c035ff797cde820da6a54366cf0710f719b7689"   # <-- remplacer

# ACS 5-Year 2022
ACS_BASE = "https://api.census.gov/data/2022/acs/acs5"

# TIGER/Web REST API — couche ZCTA 2020 (layer 4 = Feature Layer, layer 2 = Group Layer sans données)
TIGER_ZCTA_URL = (
    "https://tigerweb.geo.census.gov/arcgis/rest/services/"
    "TIGERweb/PUMA_TAD_TAZ_UGA_ZCTA/MapServer/4/query"
)

print("Librairies chargées — pandas", pd.__version__)

## 1. ZIP codes de test

Le dataset telco ne contient pas de champ zipcode. On utilise ici des ZCTA représentatifs  
de différents profils socio-économiques américains pour valider l'enrichissement.

In [ ]:
# Zipcodes représentatifs (urbain / suburbain / rural, régions variées)
SAMPLE_ZIPCODES = [
    "10001",  # Manhattan, NY         — dense / aisé
    "10451",  # Bronx, NY             — dense / modeste
    "90210",  # Beverly Hills, CA     — aisé
    "90011",  # Los Angeles, CA       — dense / modeste
    "60601",  # Chicago Loop, IL      — dense / mixte
    "60629",  # Chicago South, IL     — modeste
    "77004",  # Houston, TX           — mixte  (remplace 77001 : PO Box uniquement)
    "33101",  # Miami, FL             — mixte
    "85008",  # Phoenix, AZ           — suburbain  (remplace 85001 : PO Box uniquement)
    "98101",  # Seattle, WA           — aisé / tech
    "30308",  # Atlanta, GA           — mixte  (remplace 30301 : PO Box uniquement)
    "19104",  # Philadelphia, PA      — dense / modeste  (remplace 19101 : PO Box uniquement)
    "78201",  # San Antonio, TX       — modeste
    "97201",  # Portland, OR          — aisé
    "59001",  # Montana               — rural / faible densité
]

print(f"{len(SAMPLE_ZIPCODES)} zipcodes de test chargés")
print(SAMPLE_ZIPCODES)

## 2. Fonctions utilitaires

In [3]:
def census_get(url: str) -> dict | list | None:
    """Appel HTTP GET simple avec gestion d'erreur."""
    try:
        req = urllib.request.Request(url, headers={"User-Agent": "churn-data-platform/1.0"})
        with urllib.request.urlopen(req, timeout=15) as resp:
            raw = resp.read().decode("utf-8")
            # L'API Census renvoie parfois du HTML quand la clé est manquante
            if raw.strip().startswith("<"):
                print("[ERREUR] Réponse HTML reçue — vérifier la clé API Census")
                print(raw[:300])
                return None
            return json.loads(raw)
    except Exception as exc:
        print(f"[ERREUR] {exc}")
        return None


def build_acs_url(zipcodes: list[str], variables: list[str], api_key: str) -> str:
    """Construit l'URL ACS pour une liste de ZCTA."""
    zcta_list = ",".join(zipcodes)
    var_str   = ",".join(["NAME"] + variables)
    params = {
        "get": var_str,
        "for": f"zip code tabulation area:{zcta_list}",
        "key": api_key,
    }
    return ACS_BASE + "?" + urllib.parse.urlencode(params)


print("Fonctions utilitaires définies")

Fonctions utilitaires définies


## 3. Récupération — Revenu médian & Âge médian (ACS)

| Variable Census | Signification |
|---|---|
| `B19013_001E` | Median household income (USD, 12 derniers mois) |
| `B01002_001E` | Median age (années) |
| `B01001_001E` | Total population |

Valeur `-666666666` = donnée non disponible (Census convention).

In [4]:
ACS_VARS = ["B19013_001E", "B01002_001E", "B01001_001E"]

url = build_acs_url(SAMPLE_ZIPCODES, ACS_VARS, CENSUS_API_KEY)
print("URL appelée (clé masquée) :")
print(url.replace(CENSUS_API_KEY, "***"))

raw_acs = census_get(url)

URL appelée (clé masquée) :
https://api.census.gov/data/2022/acs/acs5?get=NAME%2CB19013_001E%2CB01002_001E%2CB01001_001E&for=zip+code+tabulation+area%3A10001%2C10451%2C90210%2C90011%2C60601%2C60629%2C77001%2C33101%2C85001%2C98101%2C30301%2C19101%2C78201%2C97201%2C59001&key=***


In [5]:
if raw_acs is None:
    print("Échec de la requête ACS — voir message d'erreur ci-dessus")
else:
    # L'API renvoie [[headers], [row1], [row2], ...]
    headers = raw_acs[0]
    rows    = raw_acs[1:]
    print(f"Headers : {headers}")
    print(f"{len(rows)} ZCTA récupérés")
    for r in rows[:3]:
        print(r)

Headers : ['NAME', 'B19013_001E', 'B01002_001E', 'B01001_001E', 'zip code tabulation area']
11 ZCTA récupérés
['ZCTA5 10001', '106509', '35.7', '27004', '10001']
['ZCTA5 10451', '34316', '34.3', '51311', '10451']
['ZCTA5 33101', '-666666666', '-666666666.0', '0', '33101']


In [6]:
def parse_acs_response(raw: list) -> pd.DataFrame:
    """Convertit la réponse ACS brute en DataFrame propre."""
    headers, rows = raw[0], raw[1:]
    df = pd.DataFrame(rows, columns=headers)

    # Renommage clair
    df = df.rename(columns={
        "zip code tabulation area": "zipcode",
        "B19013_001E": "median_household_income_usd",
        "B01002_001E": "median_age",
        "B01001_001E": "total_population",
    })

    # Cast numérique + remplacer valeurs manquantes Census (-666666666)
    num_cols = ["median_household_income_usd", "median_age", "total_population"]
    df[num_cols] = df[num_cols].apply(pd.to_numeric, errors="coerce")
    df[num_cols] = df[num_cols].where(df[num_cols] > -1, other=pd.NA)

    return df[["zipcode", "median_household_income_usd", "median_age", "total_population", "NAME"]]


if raw_acs:
    df_acs = parse_acs_response(raw_acs)
    display(df_acs.sort_values("median_household_income_usd", ascending=False))

,zipcode,median_household_income_usd,median_age,total_population,NAME
8,90210,172285.0,50.9,19180,ZCTA5 90210
4,60601,121294.0,31.8,16398,ZCTA5 60601
10,98101,111099.0,34.7,16302,ZCTA5 98101
0,10001,106509.0,35.7,27004,ZCTA5 10001
3,59001,76120.0,56.6,1337,ZCTA5 59001
9,97201,64385.0,32.3,17508,ZCTA5 97201
5,60629,53318.0,32.9,108997,ZCTA5 60629
7,90011,51819.0,30.6,106042,ZCTA5 90011
6,78201,40356.0,37.7,45355,ZCTA5 78201
1,10451,34316.0,34.3,51311,ZCTA5 10451


## 4. Récupération — Superficie terrestre (TIGER/Web API)

L'API TIGER renvoie `ALAND` (superficie terrestre en m²) pour chaque ZCTA.  
Densité = `total_population / (ALAND / 1_000_000)` → habitants par km²

In [ ]:
def fetch_tiger_area(zipcodes: list[str]) -> pd.DataFrame | None:
    """
    Récupère la superficie terrestre (AREALAND en m²) via TIGER/Web REST API.
    Couche 4 = '2020 Census ZIP Code Tabulation Areas' (Feature Layer).
    Champs : ZCTA5 (identifiant), AREALAND (superficie terrestre en m²).
    """
    all_rows = []
    batch_size = 50

    for i in range(0, len(zipcodes), batch_size):
        batch = zipcodes[i : i + batch_size]
        where_clause = "ZCTA5 IN ('" + "','".join(batch) + "')"
        params = {
            "where": where_clause,
            "outFields": "ZCTA5,AREALAND",
            "f": "json",
            "returnGeometry": "false",
        }
        url = TIGER_ZCTA_URL + "?" + urllib.parse.urlencode(params)
        print(f"Batch {i//batch_size + 1} — {len(batch)} ZCTA")

        data = census_get(url)
        if data is None:
            return None

        features = data.get("features", [])
        if not features:
            print("[WARN] Aucun feature retourné")
            print("Réponse brute :", str(data)[:400])
            continue

        for f in features:
            attrs = f["attributes"]
            all_rows.append({
                "zipcode": str(attrs.get("ZCTA5", "")).zfill(5),
                "aland_m2": attrs.get("AREALAND"),
            })

        time.sleep(0.3)

    if not all_rows:
        return None
    return pd.DataFrame(all_rows)


df_tiger = fetch_tiger_area(SAMPLE_ZIPCODES)

In [8]:
if df_tiger is not None:
    print(f"{len(df_tiger)} ZCTA avec superficie")
    display(df_tiger.head(5))
else:
    print("TIGER API non disponible — densité ne pourra pas être calculée")

TIGER API non disponible — densité ne pourra pas être calculée


## 4.1 Diagnostic — Vérification des champs disponibles dans la couche TIGER

Avant de corriger la requête, on interroge les **métadonnées de la couche** pour connaître les noms exacts des champs.

In [ ]:
# Méthode 1 — Métadonnées de la couche TIGER (URL sans /query, avec ?f=json)
# Layer 4 = Feature Layer ZCTA (layer 2 = Group Layer sans champs)
layer_meta_url = (
    "https://tigerweb.geo.census.gov/arcgis/rest/services/"
    "TIGERweb/PUMA_TAD_TAZ_UGA_ZCTA/MapServer/4?f=json"
)

meta = census_get(layer_meta_url)

if meta and meta.get("fields"):
    print(f"Couche    : {meta.get('name', 'N/A')}")
    print(f"Nb champs : {len(meta['fields'])}")
    print()
    print(f"  {'Nom du champ':<30} {'Type':<30} Alias")
    print("  " + "-" * 75)
    for f in meta["fields"]:
        print(f"  {f['name']:<30} {f['type']:<30} {f.get('alias', '')}")
else:
    print("Impossible de récupérer les métadonnées")
    print("Réponse brute :", str(meta)[:400])

## 5. Calcul de la densité & fusion finale

In [9]:
def compute_enriched_geo(df_acs: pd.DataFrame, df_tiger: pd.DataFrame | None) -> pd.DataFrame:
    """Fusionne ACS + TIGER et calcule la densité de population."""
    df = df_acs.copy()

    if df_tiger is not None:
        df_tiger["aland_km2"] = pd.to_numeric(df_tiger["aland_m2"], errors="coerce") / 1_000_000
        df = df.merge(df_tiger[["zipcode", "aland_km2"]], on="zipcode", how="left")
        df["population_density_per_km2"] = (
            df["total_population"] / df["aland_km2"]
        ).round(1)
    else:
        df["aland_km2"] = pd.NA
        df["population_density_per_km2"] = pd.NA

    # Colonnes finales pour l'enrichissement
    return df[[
        "zipcode",
        "NAME",
        "median_household_income_usd",
        "median_age",
        "total_population",
        "aland_km2",
        "population_density_per_km2",
    ]]


if 'df_acs' in dir():
    df_geo = compute_enriched_geo(df_acs, df_tiger)
    display(df_geo.sort_values("population_density_per_km2", ascending=False))
else:
    print("df_acs non disponible — relancer les cellules ACS ci-dessus")

,zipcode,NAME,median_household_income_usd,median_age,total_population,aland_km2,population_density_per_km2
0,10001,ZCTA5 10001,106509.0,35.7,27004,<NA>,<NA>
1,10451,ZCTA5 10451,34316.0,34.3,51311,<NA>,<NA>
2,33101,ZCTA5 33101,NaN,NaN,0,<NA>,<NA>
3,59001,ZCTA5 59001,76120.0,56.6,1337,<NA>,<NA>
4,60601,ZCTA5 60601,121294.0,31.8,16398,<NA>,<NA>
5,60629,ZCTA5 60629,53318.0,32.9,108997,<NA>,<NA>
6,78201,ZCTA5 78201,40356.0,37.7,45355,<NA>,<NA>
7,90011,ZCTA5 90011,51819.0,30.6,106042,<NA>,<NA>
8,90210,ZCTA5 90210,172285.0,50.9,19180,<NA>,<NA>
9,97201,ZCTA5 97201,64385.0,32.3,17508,<NA>,<NA>


## 6. Visualisation rapide

In [10]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

if 'df_geo' in dir() and df_geo['population_density_per_km2'].notna().any():
    df_plot = df_geo.dropna(subset=["median_household_income_usd", "population_density_per_km2"]).copy()
    df_plot["city"] = df_plot["NAME"].str.extract(r"ZCTA5 \d+; (.+?)(?:,|$)").fillna(df_plot["zipcode"])

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle("Enrichissement Census — Métriques par ZCTA", fontsize=14, fontweight="bold")

    # Revenu médian
    ax = axes[0]
    df_s = df_plot.sort_values("median_household_income_usd")
    ax.barh(df_s["zipcode"], df_s["median_household_income_usd"], color="steelblue")
    ax.set_title("Revenu médian (USD)")
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}"))
    ax.set_xlabel("USD")

    # Âge médian
    ax = axes[1]
    df_s = df_plot.sort_values("median_age")
    ax.barh(df_s["zipcode"], df_s["median_age"], color="coral")
    ax.set_title("Âge médian (années)")
    ax.set_xlabel("Années")

    # Densité de population
    ax = axes[2]
    df_s = df_plot.sort_values("population_density_per_km2")
    ax.barh(df_s["zipcode"], df_s["population_density_per_km2"], color="mediumseagreen")
    ax.set_title("Densité (hab/km²)")
    ax.set_xlabel("hab/km²")

    plt.tight_layout()
    plt.savefig("census_geo_enrichment_test.png", dpi=120, bbox_inches="tight")
    plt.show()
    print("Graphique sauvegardé : census_geo_enrichment_test.png")
else:
    print("Données insuffisantes pour la visualisation")

Données insuffisantes pour la visualisation


## 7. Simulation d'enrichissement sur telco.csv

Le dataset telco actuel n'a pas de colonne zipcode.  
Ce bloc simule ce que donnerait l'enrichissement si on avait un zipcode par client.

In [ ]:
import numpy as np

# Chargement du dataset telco avec zipcodes simulés réalistement
# Logique : MonthlyCharges → profil de richesse du zipcode, SeniorCitizen → biais zone rurale/retraitée
df_telco = pd.read_csv("../data/raw/telco.csv", dtype={"zipcode": str})
df_telco["zipcode"] = df_telco["zipcode"].str.zfill(5)
print(f"Telco shape : {df_telco.shape}")
print(f"Colonnes : {list(df_telco.columns)}")
print()
print("Distribution des zipcodes :")
print(df_telco["zipcode"].value_counts())
print()
print(df_telco[["customerID", "MonthlyCharges", "SeniorCitizen", "zipcode", "Churn"]].head(5))

In [ ]:
if 'df_geo' in dir():
    # Jointure avec les données Census
    df_enriched = df_telco.merge(
        df_geo.drop(columns=["NAME", "total_population", "aland_km2"]),
        on="zipcode", how="left"
    )
    print(f"Shape après enrichissement : {df_enriched.shape}")
    print(f"Colonnes ajoutées : median_household_income_usd, median_age, population_density_per_km2")
    display(df_enriched[["customerID", "zipcode", "MonthlyCharges", "Churn",
                          "median_household_income_usd", "median_age",
                          "population_density_per_km2"]].head(10))
else:
    print("df_geo non disponible — relancer les cellules Census ci-dessus")

In [13]:
# Analyse préliminaire : corrélation revenu médian vs churn
if 'df_enriched' in dir():
    df_enriched["Churn_binary"] = (df_enriched["Churn"] == "Yes").astype(int)

    churn_by_income = (
        df_enriched.groupby("zipcode")[["median_household_income_usd", "median_age",
                                        "population_density_per_km2", "Churn_binary"]]
        .mean()
        .rename(columns={"Churn_binary": "churn_rate"})
        .sort_values("median_household_income_usd")
    )

    print("Corrélations avec le taux de churn :")
    print(churn_by_income.corr()["churn_rate"].drop("churn_rate"))
    display(churn_by_income)

Corrélations avec le taux de churn :
median_household_income_usd   -0.112000
median_age                     0.450376
population_density_per_km2          NaN
Name: churn_rate, dtype: float64


,median_household_income_usd,median_age,population_density_per_km2,churn_rate
zipcode,,,,
10451,34316.0,34.3,NaN,0.293763
78201,40356.0,37.7,NaN,0.283898
90011,51819.0,30.6,NaN,0.267261
60629,53318.0,32.9,NaN,0.232346
97201,64385.0,32.3,NaN,0.251121
59001,76120.0,56.6,NaN,0.287879
10001,106509.0,35.7,NaN,0.255459
98101,111099.0,34.7,NaN,0.282460
60601,121294.0,31.8,NaN,0.248497


## 8. Fonction d'enrichissement réutilisable

Prête à intégrer dans un pipeline ETL.

In [ ]:
def enrich_with_census(
    df: pd.DataFrame,
    zipcode_col: str,
    api_key: str,
    acs_year: int = 2022,
    batch_size: int = 50,
) -> pd.DataFrame:
    """
    Enrichit un DataFrame avec les métriques Census par zipcode.

    Parameters
    ----------
    df          : DataFrame source avec une colonne zipcode
    zipcode_col : nom de la colonne zipcode
    api_key     : clé API Census (gratuite sur https://api.census.gov/data/key_signup.html)
    acs_year    : année ACS (défaut 2022)
    batch_size  : nombre de zipcodes par requête (max 500 pour Census API)

    Returns
    -------
    DataFrame enrichi avec 3 nouvelles colonnes :
        - median_household_income_usd
        - median_age
        - population_density_per_km2
    """
    acs_base = f"https://api.census.gov/data/{acs_year}/acs/acs5"
    # Layer 4 = '2020 Census ZIP Code Tabulation Areas' (Feature Layer)
    tiger_url = (
        "https://tigerweb.geo.census.gov/arcgis/rest/services/"
        "TIGERweb/PUMA_TAD_TAZ_UGA_ZCTA/MapServer/4/query"
    )
    unique_zips = df[zipcode_col].dropna().astype(str).str.zfill(5).unique().tolist()
    print(f"[Census] {len(unique_zips)} zipcodes uniques à enrichir")

    all_acs, all_tiger = [], []

    for i in range(0, len(unique_zips), batch_size):
        batch = unique_zips[i : i + batch_size]

        # — ACS —
        acs_vars = "NAME,B19013_001E,B01002_001E,B01001_001E"
        zcta_str = ",".join(batch)
        params = urllib.parse.urlencode({
            "get": acs_vars,
            "for": f"zip code tabulation area:{zcta_str}",
            "key": api_key,
        })
        raw = census_get(acs_base + "?" + params)
        if raw:
            chunk = parse_acs_response(raw)
            all_acs.append(chunk)

        # — TIGER — champ AREALAND (superficie terrestre en m²)
        where = "ZCTA5 IN ('" + "','".join(batch) + "')"
        tiger_params = urllib.parse.urlencode({
            "where": where,
            "outFields": "ZCTA5,AREALAND",
            "f": "json",
            "returnGeometry": "false",
        })
        tiger_raw = census_get(tiger_url + "?" + tiger_params)
        if tiger_raw and tiger_raw.get("features"):
            rows = [
                {
                    "zipcode": str(f["attributes"].get("ZCTA5", "")).zfill(5),
                    "aland_m2": f["attributes"].get("AREALAND"),
                }
                for f in tiger_raw["features"]
            ]
            all_tiger.extend(rows)

        time.sleep(0.2)

    df_acs_all   = pd.concat(all_acs, ignore_index=True) if all_acs else pd.DataFrame()
    df_tiger_all = pd.DataFrame(all_tiger) if all_tiger else None

    df_geo_full = compute_enriched_geo(df_acs_all, df_tiger_all) if not df_acs_all.empty else pd.DataFrame()

    keep_cols = ["zipcode", "median_household_income_usd", "median_age", "population_density_per_km2"]
    df_result = df.copy()
    df_result[zipcode_col] = df_result[zipcode_col].astype(str).str.zfill(5)
    df_result = df_result.merge(
        df_geo_full[keep_cols].rename(columns={"zipcode": zipcode_col}),
        on=zipcode_col,
        how="left",
    )

    n_enriched = df_result["median_household_income_usd"].notna().sum()
    print(f"[Census] {n_enriched}/{len(df_result)} lignes enrichies ({n_enriched/len(df_result)*100:.1f}%)")
    return df_result


print("Fonction enrich_with_census() prête")
print()
print("Usage :")
print('  df_enriched = enrich_with_census(df_telco, zipcode_col="zipcode", api_key=CENSUS_API_KEY)')

## 9. Résumé — Variables Census disponibles pour le modèle

| Colonne | Source Census | Utilité pour le modèle churn |
|---|---|---|
| `median_household_income_usd` | ACS B19013_001E | Proxy niveau de vie — impact sur la sensibilité prix |
| `median_age` | ACS B01002_001E | Profil démographique — seniors vs jeunes actifs |
| `population_density_per_km2` | ACS pop + TIGER ALAND | Urbain vs rural — offres réseau disponibles |

### Prochaines étapes
1. Ajouter un champ `zipcode` dans la collecte client
2. Intégrer `enrich_with_census()` dans le pipeline ETL (après le chargement de `telco.csv`)
3. Analyser l'importance de ces features dans le modèle de churn (SHAP / feature importance)
4. Envisager d'autres variables ACS : taux de chômage (`B23025_005E`), niveau d'éducation (`B15003_022E`)